# Stage 4: the pixels

**The question.** Does the first frame predict a video's lift, beyond who made the video and what it says? `lift` is
stage 1's target: the video's log share rate minus the median of its creator's last 30 videos posted at least 14 days
earlier. Who made the video is in the target already, B3 (stage 2: the video's length and timing) and the text model
(stage 3: seven text fields in ridge on top of B3) say the rest, and this stage asks what the frame adds to them.

Two sets of embeddings come back from one rented GPU (`docs/gpu.md`):

- **The text, a second time:** Qwen3-Embedding-4B over the seven text fields, the larger encoder notebook 03's hand-off
  sent to the rental, with its bar written there. It is scored first: the better of it and bge-small is the reference
  the pixels must beat.
- **The frames:** one per video, made in stage 4's prep (`notebooks/04_frames.ipynb`) by the platform's own poster
  code, so a kept clip's frame and a retired clip's poster can't be told apart by how they were made (trap 3).

**The image encoder: SigLIP 2, so400m, patch 14, 384 px** (`google/siglip2-so400m-patch14-384`), frozen: 1,152
numbers per frame. Why this one:

- **It learned from images and their captions,** like CLIP, so its numbers describe what a picture shows: people,
  places, objects, text. That is this stage's question. A self-supervised encoder such as DINOv2 learns what makes two
  pictures look alike (the room, the light, the camera), which is closer to who filmed the video than to what works.
- **It sees the whole frame.** Its own preprocessing squashes the 9:16 frame into a 384×384 square. CLIP's resizes the
  short side and keeps the centre square, which cuts 22% off the top and 22% off the bottom of a vertical video, where
  the hook and the captions usually sit; stage 3 found on-screen text the strongest single field.
- **It is the strongest of its kind that the rental runs cheaply.** The SigLIP 2 paper reports it ahead of the first
  SigLIP at every size, in classification and retrieval without fine-tuning, and the first SigLIP was already ahead of
  CLIP. so400m is its largest size short of the 1-billion-parameter one: about 400 million parameters in the image
  tower, so one RTX 3090 embeds every frame in float32 in about 75 minutes (estimated from the model's size, not
  measured), and 1,152 numbers keep every head on this laptop.
- **Frozen first.** It is the same model class and preprocessing as the SigLIP 2 base the GPU kit was tested with
  (`docs/gpu.md`, section 6), so the box runs code that already ran. Fine-tuning comes only if the frozen vectors pass.

**The rental, estimated before it** (`docs/gpu.md`, section 4): 2× RTX 3090, the 4B text job about 2 to 5 hours on both
cards, the frames about 40 minutes on both, about 3 to 6.5 hours and $1 to $2.50 in all. Section 1 records what it took.

The code is `src/sharerate/` (the loader, the text cache, the frames, the bootstrap and the metrics), with the heads
written out in the cells, as in notebook 03.

## 0. The bar, written before the rental (rule 7)

This cell was written, and committed, before the box was rented: no Qwen3-Embedding-4B or SigLIP 2 so400m vector
of these videos existed yet.

**Part A. The text: Qwen3-Embedding-4B against bge-small.** Notebook 03's hand-off wrote this bar before the 4B ran. It
is applied as written and never re-decided:

- Qwen3-Embedding-4B (its first 1,024 numbers, from the first 256 tokens, unit length, no instruction) replaces
  bge-small if the 95% paired creator-bootstrap interval of its ridge head's validation MAE gain over bge ridge lies
  above 0 for **seen** creators (138, 13,646 videos), **with and without the 73 reposts**. Both heads sit on top of B3,
  with notebook 03's `ridge_head`, on the same 21,332 validation rows, 2,000 draws, seed 2026. Otherwise bge-small
  stays.
- The 4B ridge's penalty is chosen on validation as bge's was, over notebook 03's grid (10 to 10,000). If the choice
  lands on an end of the grid, the grid gets one more step that way (a factor of about 3), and the notebook says so.
- The 4B's logistic head (C chosen by validation ROC-AUC, as in notebook 03) is reported, not judged. It goes to
  stage 5 only if the 4B wins.
- **The winner is the reference** for everything below: B3 ridge plus the winning text head.

**Part B. The pixels: does the frame add to the reference?**

- **The rows.** Only rows with a frame (`notebooks/04_frames.ipynb`, cell 4). Models are fitted on the 74,518 training
  rows that have one (of 74,552) and scored on the 21,328 validation rows that have one (of 21,332: 13,645 of the seen
  creators', 2,274 of the new ones', 5,409 of the held-out ones'). The reference is re-scored on exactly these rows
  (rule 5). The rows without a frame are counted, and compared in lift if they come from at least 5 creators.
- **The reference on the training rows is out of fold.** The image heads are fitted to what the reference leaves on
  the training rows. The text head has thousands of columns and was fitted on those same rows, so its residuals there
  are smaller than the ones it leaves on videos it never saw, and an image head fitted to them would learn from
  residuals that are too clean. So B3 and the text head are refitted in 5 folds grouped by creator, with their
  penalties fixed, and each training row gets the reference from a fit that never saw its creator. Validation rows get
  the full fit: the model Part A scored.
- **Image ridge.** SigLIP 2's 1,152 numbers, scaled to unit length, in ridge fitted on the training rows to what the
  reference leaves. The prediction is the reference plus the head. Its penalty is chosen by validation MAE over all
  21,328 validation rows with a frame, from 1 to 10,000 in steps of about 3, with the same one-more-step rule at an
  end.
- **Image boosting,** the small non-linear model: scikit-learn's `HistGradientBoostingRegressor` on the first 64
  principal components of the unit vectors (fitted on the training rows) and the reference's prediction, fitted to
  what the reference leaves. The prediction is the reference plus its output. Fixed now: learning rate 0.05, at most 15
  leaves a tree, at least 200 videos a leaf, an L2 penalty of 1, squared error, seed 2026. Only the number of trees (25
  to 1,000, in steps of 25) is chosen, by validation MAE. **Its control** is the same model on the reference's
  prediction alone: boosting can bend the reference by itself, and that bend is not the frame's. The boosting's gain
  is measured over its control.
- **The primary bar.** An image model beats the reference if the 95% paired creator-bootstrap interval (2,000 draws,
  seed 2026) of its validation MAE gain over its base (the reference for the ridge, the control for the boosting) lies
  above 0 for **seen** creators, **with and without the reposts**.
- **Does a frame name its creator? This decides the metric.** A logistic regression on the standardised unit vectors
  (C 1, fixed now), fitted on the training frames to name each frame's creator (148 creators), names the creator of
  each seen creator's validation frame. If its top-1 accuracy is at least 5 times what always guessing the biggest
  creator scores, frames identify their creators well, and a pixel model could gain by learning who made a video,
  which lift already accounts for. Then a pass also needs the gain in Spearman inside a creator (over the same base,
  seen creators) to have an interval above 0: a model that only knew the creator would rank nothing inside one.
- **The leak check, stage 3's, on the pixels.** Inside each creator, the image model's correction (its prediction
  minus its base's) is regressed on whether the platform kept the clip and on the lift (notebook 03's
  `kept_given_lift`, seen creators). If the kept flag's interval lies above 0, the model finds kept clips by something
  other than the lift, and it is reported as a leak, not a pass, whatever its gain. Spearman inside a creator among
  kept clips alone and among retired clips alone is reported beside it.
- **Reported, not required:** new and held-out creators; ROC-AUC, PR-AUC and the top-10% share.
- **Fine-tuning runs only if a frozen image model passes** everything above. Then a second rental fine-tunes SigLIP 2's
  image tower, or a light adapter on it, on the training rows to the same target, stopping when validation MAE stops
  improving, with every run logged to an MLflow file store. It replaces the frozen model only if its MAE gain over that
  model has an interval above 0 for seen creators, with and without the reposts.
- **What goes to stage 5.** Part A's text winner, and the passing image model with the largest MAE gain for seen
  creators. If no image model passes, stage 5 leaves the pixels out, and this notebook says the first frame adds
  nothing beyond who made the video and what it says. That result is publishable.
- **Fixed in advance, never tuned:** the encoder and its preprocessing (float32, the whole frame squashed to 384×384 by
  its own processor), unit length, the 64 components, the boosting's settings, the creator probe's, the 5 folds, the
  seeds. **Chosen on validation:** the ridge penalties, the logistic C and the number of trees. The test split stays
  closed; stage 5 opens it once.
- **A miss is a miss.** A gain whose interval touches 0 is reported as a miss, never tuned until it passes.